In [2]:
# Mount Drive (in case it disconnected)
from google.colab import drive
drive.mount('/content/drive')

# Verify the path exists
from pathlib import Path
DRIVE_RAW = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw")
print(f"Drive raw exists: {DRIVE_RAW.exists()}")

# Now save the dataframe (df is still in memory from your previous cell)
output_path = DRIVE_RAW / "rq4_cost_dimensions_v2.csv"
df.to_csv(output_path, index=False)

print(f"\n✓ Saved: {output_path.name}")
print(f"  {len(df)} rows × {len(df.columns)} cols")
print(f"\nIndicator coverage:")
print(df.groupby('indicator')['country_iso3'].nunique().to_string())
print(f"\nINDIA SNAPSHOT (most recent year per indicator):")
india_recent = df[df['country_iso3'] == 'IND'].sort_values('year').drop_duplicates('indicator', keep='last')
for _, row in india_recent.iterrows():
    print(f"  {row['indicator']:30s} {int(row['year'])}: {row['value']:.2f}")

Mounted at /content/drive
Drive raw exists: True

✓ Saved: rq4_cost_dimensions_v2.csv
  380 rows × 8 cols

Indicator coverage:
indicator
gdp_per_capita_usd        10
lending_rate_pct           9
real_interest_rate_pct     9
td_losses_pct             10

INDIA SNAPSHOT (most recent year per indicator):
  real_interest_rate_pct         2022: 2.52
  lending_rate_pct               2022: 8.57
  td_losses_pct                  2023: 14.16
  gdp_per_capita_usd             2024: 2694.74


In [3]:
# ====================================================================
# RQ4 SUPPLEMENTARY EXTRACTION — proper semiconductor cost dimensions
# Standalone cell - does not modify Script 1 structure
# ====================================================================

import requests
import pandas as pd
from pathlib import Path
from datetime import datetime
import time

DRIVE_RAW = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw")

# Cost-relevant World Bank indicators
COST_INDICATORS = {
    'FR.INR.LEND': 'lending_rate_pct',
    'FR.INR.RINR': 'real_interest_rate_pct',
    'IC.TAX.TOTL.CP.ZS': 'total_tax_rate_pct',
    'IC.TAX.PRFT.CP.ZS': 'profit_tax_rate_pct',
    'IC.TAX.LABR.CP.ZS': 'labor_tax_rate_pct',
    'EG.ELC.LOSS.ZS': 'td_losses_pct',
    'NY.GDP.PCAP.CD': 'gdp_per_capita_usd',
}

COUNTRIES = ['IND', 'CHN', 'KOR', 'TWN', 'MYS', 'VNM', 'JPN', 'DEU', 'USA', 'SGP', 'THA']

print("=" * 70)
print("RQ4 SUPPLEMENTARY EXTRACTION — proper semiconductor cost dimensions")
print("=" * 70)

all_data = []
for indicator_code, indicator_name in COST_INDICATORS.items():
    countries_str = ';'.join(COUNTRIES)
    url = f"https://api.worldbank.org/v2/country/{countries_str}/indicator/{indicator_code}?format=json&date=2014:2024&per_page=500"

    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            data = response.json()
            if len(data) > 1 and data[1]:
                count = 0
                for record in data[1]:
                    if record.get('value') is not None:
                        all_data.append({
                            'country_iso3': record['countryiso3code'],
                            'country': record['country']['value'],
                            'year': int(record['date']),
                            'indicator_code': indicator_code,
                            'indicator': indicator_name,
                            'value': record['value'],
                        })
                        count += 1
                print(f"✓ {indicator_name}: {count} records")
            else:
                print(f"⚠ {indicator_name}: No data returned")
        else:
            print(f"✗ {indicator_name}: HTTP {response.status_code}")
    except Exception as e:
        print(f"✗ {indicator_name}: {e}")

    time.sleep(0.5)

df = pd.DataFrame(all_data)
df['data_source'] = 'WorldBank_API_real_supplementary'
df['retrieval_date'] = datetime.now().strftime('%Y-%m-%d')

output_path = DRIVE_RAW / "rq4_cost_dimensions_v2.csv"
df.to_csv(output_path, index=False)

print(f"\n{'='*70}")
print(f"✓ Saved: {output_path.name}")
print(f"  {len(df)} rows × {len(df.columns)} cols")
print(f"\nIndicator coverage:")
print(df.groupby('indicator')['country_iso3'].nunique().to_string())
print(f"\nINDIA SNAPSHOT (most recent year per indicator):")
india_recent = df[df['country_iso3'] == 'IND'].sort_values('year').drop_duplicates('indicator', keep='last')
for _, row in india_recent.iterrows():
    print(f"  {row['indicator']:30s} {int(row['year'])}: {row['value']:.2f}")

RQ4 SUPPLEMENTARY EXTRACTION — proper semiconductor cost dimensions
✓ lending_rate_pct: 83 records
✓ real_interest_rate_pct: 83 records
⚠ total_tax_rate_pct: No data returned
⚠ profit_tax_rate_pct: No data returned
⚠ labor_tax_rate_pct: No data returned
✓ td_losses_pct: 104 records
✓ gdp_per_capita_usd: 110 records

✓ Saved: rq4_cost_dimensions_v2.csv
  380 rows × 8 cols

Indicator coverage:
indicator
gdp_per_capita_usd        10
lending_rate_pct           9
real_interest_rate_pct     9
td_losses_pct             10

INDIA SNAPSHOT (most recent year per indicator):
  real_interest_rate_pct         2022: 2.52
  lending_rate_pct               2022: 8.57
  td_losses_pct                  2023: 14.16
  gdp_per_capita_usd             2024: 2694.74


In [6]:
# ====================================================================
# FIGURE 5 (REGENERATED) — Cost Dimensions Heatmap with proper variables
# ====================================================================

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

# COLORS palette (re-defined for standalone execution)
COLORS = {
    'primary':   '#2E5C8A',
    'secondary': '#1C7293',
    'accent':    '#E07A5F',
    'neutral':   '#606060',
    'light':     '#D0D0D0',
    'success':   '#5C8A5C',
    'warning':   '#D4A85C',
    'india':     '#FF8C42',
}

DRIVE_RAW = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw")
DRIVE_FIG = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/figures")

# Read the new cost dimensions file
df = pd.read_csv(DRIVE_RAW / "rq4_cost_dimensions_v2.csv", low_memory=False)
print(f"Loaded: {len(df)} rows, indicators: {df['indicator'].unique().tolist()}")

# Country labels
country_label_map = {
    'IND': 'India', 'CHN': 'China', 'KOR': 'S. Korea', 'TWN': 'Taiwan',
    'MYS': 'Malaysia', 'VNM': 'Vietnam', 'JPN': 'Japan', 'DEU': 'Germany',
    'USA': 'USA', 'SGP': 'Singapore', 'THA': 'Thailand',
}

# Indicator labels (semiconductor-relevant cost names)
indicator_labels = {
    'lending_rate_pct': 'Lending Rate (%)',
    'real_interest_rate_pct': 'Real Interest Rate (%)',
    'td_losses_pct': 'T&D Losses (%)',
    'gdp_per_capita_usd': 'GDP per capita (USD)',
}

# Get most recent year per country per indicator
latest = df.sort_values('year').drop_duplicates(['country_iso3', 'indicator'], keep='last')

# Pivot to country × indicator
pivot = latest.pivot_table(index='country_iso3', columns='indicator', values='value', aggfunc='first')
pivot = pivot[list(indicator_labels.keys())]

# Apply labels
pivot.index = [country_label_map.get(idx, idx) for idx in pivot.index]
pivot.columns = [indicator_labels[c] for c in pivot.columns]

# Filter to comparator countries
target = ['India', 'China', 'S. Korea', 'Malaysia', 'Vietnam', 'Japan', 'Germany', 'USA', 'Singapore', 'Thailand']
pivot = pivot.reindex([c for c in target if c in pivot.index])

print(f"\nPivot table:")
print(pivot.round(2))

# Z-score normalize each cost column for visual comparability
pivot_norm = (pivot - pivot.mean()) / pivot.std()

# Apply font defaults
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'axes.spines.top': False, 'axes.spines.right': False,
})

# Build figure
fig, ax = plt.subplots(figsize=(10, 5.5))

cmap = LinearSegmentedColormap.from_list('costmap',
    [COLORS['secondary'], '#FFFFFF', COLORS['accent']], N=256)

im = ax.imshow(pivot_norm.values, cmap=cmap, aspect='auto', vmin=-2, vmax=2)

ax.set_xticks(range(len(pivot_norm.columns)))
ax.set_xticklabels(pivot_norm.columns, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot_norm.index)))
ax.set_yticklabels(pivot_norm.index, fontsize=9)

# Annotate cells with raw values
for i in range(len(pivot_norm.index)):
    for j in range(len(pivot_norm.columns)):
        raw_val = pivot.iloc[i, j]
        if pd.notna(raw_val):
            norm_val = pivot_norm.iloc[i, j]
            text_color = 'white' if abs(norm_val) > 1.2 else 'black'
            if raw_val > 1000:
                val_str = f'{raw_val:,.0f}'
            elif raw_val > 10:
                val_str = f'{raw_val:.1f}'
            else:
                val_str = f'{raw_val:.2f}'
            ax.text(j, i, val_str, ha='center', va='center', fontsize=8, color=text_color)

cbar = plt.colorbar(im, ax=ax, shrink=0.7)
cbar.set_label('Standardized Score (z)', fontsize=9, fontweight='bold')

ax.set_title('Semiconductor Cost Dimensions Across Countries (Most Recent Available)', pad=15, fontsize=11)
ax.set_xlabel('Cost Dimension', fontweight='bold')
ax.set_ylabel('Country', fontweight='bold')

# Highlight India row
if 'India' in pivot_norm.index:
    india_idx = pivot_norm.index.get_loc('India')
    ax.add_patch(plt.Rectangle((-0.5, india_idx - 0.5), len(pivot_norm.columns), 1,
                                fill=False, edgecolor=COLORS['india'], linewidth=2.5))

plt.tight_layout()

output_path = DRIVE_FIG / "figure_05_cost_dimensions_heatmap.png"
fig.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

print(f"\n✓ Saved: {output_path.name}")

Loaded: 380 rows, indicators: ['lending_rate_pct', 'real_interest_rate_pct', 'td_losses_pct', 'gdp_per_capita_usd']

Pivot table:
           Lending Rate (%)  Real Interest Rate (%)  T&D Losses (%)  \
India                  8.57                    2.52           14.16   
China                  4.35                    5.09            3.43   
S. Korea               4.73                    0.63            3.29   
Malaysia               5.28                    4.46            6.90   
Vietnam                9.32                    7.08            6.58   
Japan                  0.99                    1.07            4.93   
Germany                 NaN                     NaN            5.12   
USA                    3.25                   -1.09            5.31   
Singapore              5.25                   -5.12            0.23   
Thailand               4.51                    3.55            7.16   

           GDP per capita (USD)  
India                   2694.74  
China               

In [7]:
# ====================================================================
# FIGURE 6 (REGENERATED) — Equal-Weighted CCI with proper variables
# ====================================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# COLORS palette (re-defined for standalone)
COLORS = {
    'primary':   '#2E5C8A',
    'secondary': '#1C7293',
    'accent':    '#E07A5F',
    'neutral':   '#606060',
    'light':     '#D0D0D0',
    'success':   '#5C8A5C',
    'warning':   '#D4A85C',
    'india':     '#FF8C42',
}

DRIVE_RAW = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw")
DRIVE_FIG = Path("/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/figures")

# Re-read and rebuild pivot (so this cell is self-contained)
df = pd.read_csv(DRIVE_RAW / "rq4_cost_dimensions_v2.csv", low_memory=False)

country_label_map = {
    'IND': 'India', 'CHN': 'China', 'KOR': 'S. Korea', 'TWN': 'Taiwan',
    'MYS': 'Malaysia', 'VNM': 'Vietnam', 'JPN': 'Japan', 'DEU': 'Germany',
    'USA': 'USA', 'SGP': 'Singapore', 'THA': 'Thailand',
}

indicator_labels = {
    'lending_rate_pct': 'Lending Rate (%)',
    'real_interest_rate_pct': 'Real Interest Rate (%)',
    'td_losses_pct': 'T&D Losses (%)',
    'gdp_per_capita_usd': 'GDP per capita (USD)',
}

latest = df.sort_values('year').drop_duplicates(['country_iso3', 'indicator'], keep='last')
pivot = latest.pivot_table(index='country_iso3', columns='indicator', values='value', aggfunc='first')
pivot = pivot[list(indicator_labels.keys())]
pivot.index = [country_label_map.get(idx, idx) for idx in pivot.index]
pivot.columns = [indicator_labels[c] for c in pivot.columns]

target = ['India', 'China', 'S. Korea', 'Malaysia', 'Vietnam', 'Japan', 'Germany', 'USA', 'Singapore', 'Thailand']
pivot = pivot.reindex([c for c in target if c in pivot.index])

# Build CCI from cost dimensions only (exclude GDP per capita as it's wealth, not cost)
cci_cost_cols = ['Lending Rate (%)', 'Real Interest Rate (%)', 'T&D Losses (%)']
cci_data = pivot[cci_cost_cols].dropna(thresh=2).copy()

# Z-score normalize each cost dimension
for col in cci_cost_cols:
    if col in cci_data.columns and cci_data[col].std() > 0:
        cci_data[col] = (cci_data[col] - cci_data[col].mean()) / cci_data[col].std()

# Equal-weighted CCI
cci_equal = cci_data.mean(axis=1, skipna=True)

print(f"\nEqual-Weighted CCI (semiconductor cost dimensions):")
print(cci_equal.sort_values(ascending=False).round(3).to_string())

# Apply font defaults
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'axes.spines.top': False, 'axes.spines.right': False,
})

# Build bar chart
fig, ax = plt.subplots(figsize=(10, 5.5))

cci_sorted = cci_equal.sort_values()

bar_colors = [COLORS['india'] if c == 'India' else COLORS['primary'] for c in cci_sorted.index]
bar_alpha = [1.0 if c == 'India' else 0.7 for c in cci_sorted.index]

bars = ax.barh(cci_sorted.index, cci_sorted.values, color=bar_colors, edgecolor='white', linewidth=0.5)
for bar, alpha in zip(bars, bar_alpha):
    bar.set_alpha(alpha)

# Reference line at zero
ax.axvline(x=0, color=COLORS['neutral'], linewidth=1, linestyle='-')
ax.text(0, len(cci_sorted) - 0.3, 'Group Mean', fontsize=8, color=COLORS['neutral'],
        ha='left', va='bottom', style='italic')

# Annotate bars
for i, val in enumerate(cci_sorted.values):
    ha = 'left' if val >= 0 else 'right'
    offset = 0.03 if val >= 0 else -0.03
    is_india = cci_sorted.index[i] == 'India'
    ax.text(val + offset, i, f'{val:+.2f}',
            ha=ha, va='center', fontsize=8, fontweight='bold' if is_india else 'normal')

# Info box
ax.text(0.02, 0.98,
        'Equal-Weighted CCI from 3 cost dimensions:\n'
        '• Lending Rate (%)\n'
        '• Real Interest Rate (%)\n'
        '• T&D Losses (%)\n\n'
        'Higher z-score = higher costs',
        transform=ax.transAxes, fontsize=8, va='top', ha='left',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=COLORS['light']))

ax.set_xlabel('Composite Cost Index (z-score)', fontweight='bold')
ax.set_title('Composite Cost Index (CCI) — Semiconductor Cost Dimensions by Country', pad=15, fontsize=11)

# PCA note
ax.text(0.98, 0.02,
        'Note: Equal-weighted version shown.\nPCA-weighted CCI in Script 5.',
        transform=ax.transAxes, fontsize=8, style='italic', color=COLORS['neutral'],
        ha='right', va='bottom')

plt.tight_layout()

output_path = DRIVE_FIG / "figure_06_composite_cost_index_by_country.png"
fig.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

print(f"\n✓ Saved: {output_path.name}")


Equal-Weighted CCI (semiconductor cost dimensions):
India        1.226
Vietnam      1.082
Malaysia     0.338
Thailand     0.177
China       -0.028
S. Korea    -0.397
USA         -0.573
Japan       -0.706
Singapore   -1.119

✓ Saved: figure_06_composite_cost_index_by_country.png
